# Fine-tune MuRIL for ScamShield on Google Colab

1. In Colab, choose **Runtime > Change runtime type > T4 GPU** (or any GPU).
2. Run the cells from top to bottom.
3. Download `muril.zip` at the end and unzip it into `ml/models/muril/` on your machine.

Only anonymised data should be uploaded to Colab. Run `python -m ml.prepare_data` locally first and upload the files from `data/processed/`, or upload raw files and let the notebook anonymise them (raw data then passes through Google's servers).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the project code

Either clone your repository (option A) or upload a zip of the project folder (option B).

In [ ]:
# Option A: clone (replace with your repository URL; private repos need a token)
# !git clone https://github.com/<your-user>/ScamShield.git

# Option B: upload a zip of the project (without .venv) and unzip it
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "$zip_name" -d /content

In [ ]:
%cd /content/ScamShield
!pip install -q -r requirements-ml.txt

## 2. Prepare the data

If you uploaded the project with `data/processed/train.csv` and `test.csv` already created, skip this cell. Otherwise this anonymises and splits the CSV files listed in `--input`.

In [ ]:
!python -m ml.prepare_data --input data/seed_sample.csv

## 3. Train

`--fp16` uses mixed precision on the GPU. On a T4, batch size 32 fits comfortably with `--max-length 128`.

In [ ]:
!python -m ml.train_muril --device cuda --fp16 --batch-size 32 --grad-accum 1 --epochs 5

## 4. Compare with the baseline

In [ ]:
!python -m ml.train_baseline
!python -m ml.evaluate
!cat ml/results/comparison.md

## 5. Download the trained model and results

In [ ]:
!cd ml/models && zip -q -r /content/muril.zip muril
!cd ml && zip -q -r /content/results.zip results
files.download("/content/muril.zip")
files.download("/content/results.zip")